In [ ]:
# ── CELL 1: GPU + dependencies ───────────────────────────────────────────────
!nvidia-smi
!pip install pytorch-metric-learning yacs -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/results/checkpoints', exist_ok=True)
print('Ready.')

In [ ]:
# ── CELL 3: Dataset download ──────────────────────────────────────────────────
# Skip if already present
import os, glob

candidates = glob.glob('/content/data/**/PartAnnotation', recursive=True)
if candidates:
    DATA_ROOT = candidates[0]
    print('Dataset already present:', DATA_ROOT)
else:
    KAGGLE_USERNAME = "PASTE_YOUR_USERNAME"   # <── paste here
    KAGGLE_KEY      = "PASTE_YOUR_KEY"        # <── paste here

    import json
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

    os.makedirs('/content/data', exist_ok=True)
    os.chdir('/content/data')
    !pip install kaggle -q
    !kaggle datasets download -d majdouline20/shapenetpart-dataset
    !unzip -q shapenetpart-dataset.zip
    os.chdir('/content')

    candidates = glob.glob('/content/data/**/PartAnnotation', recursive=True)
    DATA_ROOT = candidates[0] if candidates else '/content/data/PartAnnotation'

print('DATA_ROOT:', DATA_ROOT)

In [ ]:
# ── CELL 4: Dataset loader ────────────────────────────────────────────────────
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SYNSET_TO_CLASS = {
    '02691156':0,'02773838':1,'02954340':2,'02958343':3,'03001627':4,
    '03261776':5,'03467517':6,'03624134':7,'03636649':8,'03642806':9,
    '03790512':10,'03797390':11,'03948459':12,'04099429':13,'04225987':14,'04379243':15
}

def load_pts(path):
    pts = []
    with open(path) as f:
        for line in f:
            v = line.strip().split()
            if len(v) >= 3:
                pts.append([float(x) for x in v[:3]])
    return np.array(pts, dtype='float32')

def load_seg(path):
    segs = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s: segs.append(int(s))
    return np.array(segs, dtype='int64')

class ShapeNet_coseg(Dataset):
    def __init__(self, partition='train', num_points=1024, obj_class=4,
                 data_root=None, train_ratio=0.8):
        self.num_points = num_points
        if data_root is None: data_root = DATA_ROOT

        target_syn = next((s for s,c in SYNSET_TO_CLASS.items() if c==obj_class), None)
        syn_dir = os.path.join(data_root, target_syn)
        pts_dir = os.path.join(syn_dir, 'points')
        seg_dir = os.path.join(syn_dir, 'points_label')

        all_ids = sorted([f[:-4] for f in os.listdir(pts_dir) if f.endswith('.pts')])
        seg_map = {}
        for dirpath, _, files in os.walk(seg_dir):
            for f in files:
                if f.endswith('.seg') and f[:-4] not in seg_map:
                    seg_map[f[:-4]] = os.path.join(dirpath, f)

        valid_ids = [i for i in all_ids if i in seg_map]
        np.random.seed(42)
        perm  = np.random.permutation(len(valid_ids))
        split = int(len(valid_ids) * train_ratio)
        chosen = [valid_ids[i] for i in (perm[:split] if partition=='train' else perm[split:])]
        self.samples = [(os.path.join(pts_dir, sid+'.pts'), seg_map[sid]) for sid in chosen]
        print(f'[ShapeNet] {partition}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        pc  = load_pts(self.samples[idx][0])
        seg = load_seg(self.samples[idx][1])
        n   = min(len(pc), len(seg))
        pc, seg = pc[:n], seg[:n]
        N   = len(pc)
        idx_s = np.random.choice(N, self.num_points, replace=(N < self.num_points))
        pc, seg = pc[idx_s], seg[idx_s]
        pc -= pc.mean(0)
        s = np.max(np.linalg.norm(pc, axis=1))
        if s > 0: pc /= s
        binary = (seg > seg.min()).astype('int64')
        return pc.astype('float32'), binary

# test
ds_tmp = ShapeNet_coseg('test', 1024, 4)
pc0, lb0 = ds_tmp[0]
print(f'Sample shape: {pc0.shape}, fg%: {lb0.mean()*100:.1f}%')

In [ ]:
# ── CELL 5: Full model definition ─────────────────────────────────────────────
import torch.nn as nn
import torch.nn.functional as F

def knn_graph(x, k):
    B,D,N = x.shape
    xt = x.permute(0,2,1)
    dist = torch.cdist(xt, xt)
    return dist.topk(k+1, dim=-1, largest=False).indices[:,:,1:]

def get_edge_feature(x, idx):
    B,D,N = x.shape; k = idx.shape[2]
    xt   = x.permute(0,2,1).contiguous()
    flat = idx.reshape(B,-1)
    nb   = torch.gather(xt,1,flat.unsqueeze(-1).expand(B,N*k,D)).view(B,N,k,D)
    xi   = xt.unsqueeze(2).expand(B,N,k,D)
    return torch.cat([xi, nb-xi], dim=-1).permute(0,3,1,2)

class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=20):
        super().__init__()
        self.k   = k
        self.net = nn.Sequential(
            nn.Conv2d(in_ch*2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2))
    def forward(self, x):
        return self.net(get_edge_feature(x, knn_graph(x, self.k))).max(dim=-1)[0]

class DGCNN(nn.Module):
    def __init__(self, k=20, emb_dim=512):
        super().__init__()
        self.ec1 = EdgeConv(3,   64,  k)
        self.ec2 = EdgeConv(64,  64,  k)
        self.ec3 = EdgeConv(64,  128, k)
        self.ec4 = EdgeConv(128, 256, k)
        self.proj = nn.Sequential(
            nn.Conv1d(64+64+128+256, emb_dim, 1, bias=False),
            nn.BatchNorm1d(emb_dim), nn.LeakyReLU(0.2), nn.Dropout(0.4))
    def forward(self, x):
        f1=self.ec1(x); f2=self.ec2(f1); f3=self.ec3(f2); f4=self.ec4(f3)
        return self.proj(torch.cat([f1,f2,f3,f4],dim=1))

class PointSampler(nn.Module):
    def __init__(self, in_dim, n_sample):
        super().__init__()
        self.n   = n_sample
        self.net = nn.Sequential(
            nn.Linear(in_dim,256), nn.ReLU(),
            nn.Linear(256,128),    nn.ReLU(),
            nn.Linear(128,1))
    def forward(self, feats):
        B,D,N = feats.shape
        scores = self.net(feats.permute(0,2,1)).squeeze(-1)
        w      = torch.softmax(scores, dim=-1)
        idx    = w.topk(self.n, dim=-1).indices
        return torch.gather(feats,2,idx.unsqueeze(1).expand(B,D,self.n)), idx, w

class PartHead(nn.Module):
    def __init__(self, in_dim, n_parts=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_parts)
    def forward(self, feats):
        return torch.softmax(self.fc(feats.permute(0,2,1)), dim=-1)

class CoSegNet(nn.Module):
    def __init__(self, n_fg=256, n_bg=256, k=20, emb_dim=512, n_parts=2):
        super().__init__()
        self.encoder    = DGCNN(k=k, emb_dim=emb_dim)
        self.fg_sampler = PointSampler(emb_dim, n_fg)
        self.bg_sampler = PointSampler(emb_dim, n_bg)
        self.part_head  = PartHead(emb_dim, n_parts)
    def forward(self, xyz):
        feats = self.encoder(xyz.permute(0,2,1))
        fg_f, fg_idx, fg_w = self.fg_sampler(feats)
        bg_f, bg_idx, bg_w = self.bg_sampler(feats)
        probs = self.part_head(feats)
        return feats, fg_f, bg_f, fg_w, bg_w, probs

print('Model defined.')

In [ ]:
# ── CELL 6: Loss functions ─────────────────────────────────────────────────────
from pytorch_metric_learning.losses import NTXentLoss
ntxent = NTXentLoss(temperature=0.07)

def contrastive_loss(fg_f, bg_f):
    B,D,_ = fg_f.shape
    fg_obj = fg_f.mean(dim=2)
    bg_obj = bg_f.mean(dim=2)
    emb    = torch.cat([fg_obj, bg_obj], dim=0)
    labels = torch.cat([torch.arange(B), torch.arange(B)]).to(fg_f.device)
    try:    return ntxent(emb, labels)
    except: return torch.tensor(0.0, device=fg_f.device)

def repulsion_loss(fg_f, bg_f):
    return F.cosine_similarity(fg_f.mean(2), bg_f.mean(2), dim=-1).mean()

def spatial_loss_uniform(feats, xyz, k=10):
    B,D,N = feats.shape
    knn_idx = torch.cdist(xyz,xyz).topk(k+1,dim=-1,largest=False).indices[:,:,1:]
    ft  = feats.permute(0,2,1).contiguous()
    flat = knn_idx.reshape(B,-1)
    nb   = torch.gather(ft,1,flat.unsqueeze(-1).expand(B,N*k,D)).view(B,N,k,D)
    fi   = ft.unsqueeze(2).expand(B,N,k,D)
    return ((fi-nb)**2).sum(-1).mean()

def entropy_loss(probs):
    return -(probs * torch.log(probs + 1e-8)).sum(-1).mean()

print('Loss functions defined.')

In [ ]:
# ── CELL 7: Training function (used once to get the best checkpoint) ───────────
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import jaccard_score, f1_score

CFG = {
    'obj_class': 4, 'num_points': 1024, 'batch_size': 8,
    'n_fg': 256, 'n_bg': 256, 'n_parts': 2, 'emb_dim': 512, 'dgcnn_k': 20
}

def train_best_model(n_epochs=15, lr=3e-4,
                     lambda_sp=0.01, k_sp=10,
                     lambda_ent=0.0001,
                     ckpt_path='/content/results/checkpoints/best_model.pt'):
    """
    Trains the best configuration (uniform spatial + entropy)
    and saves the checkpoint.
    """
    print(f'Training best model  (spatial λ={lambda_sp} k={k_sp} | entropy λ={lambda_ent} | {n_epochs} epochs)')

    train_loader = DataLoader(
        ShapeNet_coseg('train', CFG['num_points'], CFG['obj_class']),
        batch_size=CFG['batch_size'], shuffle=True, drop_last=True, num_workers=2)
    test_loader = DataLoader(
        ShapeNet_coseg('test', CFG['num_points'], CFG['obj_class']),
        batch_size=CFG['batch_size'], shuffle=False, num_workers=2)

    model = CoSegNet(CFG['n_fg'], CFG['n_bg'], CFG['dgcnn_k'], CFG['emb_dim'], CFG['n_parts']).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = StepLR(opt, step_size=15, gamma=0.5)

    best_iou = 0.0

    for epoch in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xyz, _ in train_loader:
            xyz = xyz.to(DEVICE)
            feats, fg_f, bg_f, fg_w, bg_w, probs = model(xyz)
            l_cont = contrastive_loss(fg_f, bg_f)
            l_rep  = repulsion_loss(fg_f, bg_f)
            l_sp   = spatial_loss_uniform(feats, xyz, k=k_sp)
            l_ent  = entropy_loss(probs)
            loss   = l_cont + 0.5*l_rep + lambda_sp*l_sp + lambda_ent*l_ent
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        sched.step()

        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for xyz, lbl in test_loader:
                xyz = xyz.to(DEVICE)
                feats, fg_f, bg_f, _, _, _ = model(xyz)
                ft   = feats.permute(0,2,1)
                fg_p = fg_f.mean(2, keepdim=True).permute(0,2,1)
                bg_p = bg_f.mean(2, keepdim=True).permute(0,2,1)
                pred = ((ft-fg_p).norm(-1) < (ft-bg_p).norm(-1)).long()
                preds_all.append(pred.cpu().numpy().flatten())
                labels_all.append(lbl.numpy().flatten())
        y_pred = np.concatenate(preds_all)
        y_true = np.concatenate(labels_all)
        iou = jaccard_score(y_true, y_pred, average='binary', zero_division=0)
        f1  = f1_score(y_true, y_pred, average='binary', zero_division=0)
        avg_loss = ep_loss / len(train_loader)

        marker = ''
        if iou > best_iou:
            best_iou = iou
            torch.save(model.state_dict(), ckpt_path)
            marker = '  --> Best saved'
        print(f'E{epoch:02d} | loss {avg_loss:.4f} | IOU {iou:.4f} | F1 {f1:.4f}{marker}')

    print(f'\nTraining done. Best IOU={best_iou:.4f}  Checkpoint: {ckpt_path}')
    return model, best_iou

print('train_best_model() defined.')

In [ ]:
# ── CELL 8: Train the best model to get checkpoint ────────────────────────────
# This is the ONLY training run in this notebook (~20 minutes).
# Best config confirmed from v5: uniform spatial λ=0.01 k=10 + entropy λ=0.0001

CKPT_PATH = '/content/results/checkpoints/best_model.pt'

best_model, train_iou = train_best_model(
    n_epochs=15,
    lambda_sp=0.01,
    k_sp=10,
    lambda_ent=0.0001,
    ckpt_path=CKPT_PATH
)
print(f'\nCheckpoint saved to: {CKPT_PATH}')

In [ ]:
# ── CELL 9: Inject ALL saved results from previous sessions ───────────────────
# Nothing trains here. These are your hard-saved numbers.

saved = {
    # --- v5 core results ---
    'baseline':           {'iou': 0.4031, 'f1': 0.6050},
    'sp_lam_0.001':       {'iou': 0.4528, 'f1': 0.6561},
    'sp_lam_0.005':       {'iou': 0.4322, 'f1': 0.6350},
    'sp_lam_0.010':       {'iou': 0.4568, 'f1': 0.6599},
    'sp_lam_0.100':       {'iou': 0.3802, 'f1': 0.5801},
    'k_5':                {'iou': 0.4649, 'f1': 0.6680},
    'k_10':               {'iou': 0.4568, 'f1': 0.6599},
    'k_20':               {'iou': 0.4683, 'f1': 0.6712},
    'weighted_sp_sig1':   {'iou': 0.3877, 'f1': 0.5880},
    'entropy_0.0001_15ep':{'iou': 0.4885, 'f1': 0.6906},  # BEST OVERALL
    'entropy_0.001_15ep': {'iou': 0.4631, 'f1': 0.6660},
    'full_model':         {'iou': 0.3900, 'f1': 0.5905},
    # --- v6 Task 1: sigma ablation ---
    'sigma_0.05':         {'iou': 0.3581, 'f1': 0.5273},
    'sigma_0.1':          {'iou': 0.3611, 'f1': 0.5306},
    'sigma_0.5':          {'iou': 0.3818, 'f1': 0.5526},
    'sigma_1.0':          {'iou': 0.3887, 'f1': 0.5598},  # best sigma
    'sigma_2.0':          {'iou': 0.3464, 'f1': 0.5145},
    # --- v6 Task 2: entropy 50 epochs ---
    'entropy_0.0001_50ep':{'iou': 0.3864, 'f1': 0.5574},
    # --- v6 Task 3: balance loss ---
    'balance_0.001':      {'iou': 0.3639, 'f1': 0.5336},
    'balance_0.01':       {'iou': 0.3831, 'f1': 0.5540},
    'balance_0.05':       {'iou': 0.3522, 'f1': 0.5210},
    'balance_0.1':        {'iou': 0.3533, 'f1': 0.5221},
    # --- v6 Task 4: L2 normalization ---
    'no_l2norm':          {'iou': 0.3875, 'f1': 0.5586},
    'with_l2norm':        {'iou': 0.3504, 'f1': 0.5190},
}

print('All saved results loaded.')
print(f'\nBest result: entropy_0.0001_15ep  IOU={saved["entropy_0.0001_15ep"]["iou"]:.4f}')

In [ ]:
# ── CELL 10: Robustness measurement function ──────────────────────────────────
from sklearn.metrics import jaccard_score
import numpy as np

def predict_on_cloud(model, pc_tensor):
    """
    Run inference on a single point cloud.
    pc_tensor: (1, N, 3)  on DEVICE
    returns: pred (N,) numpy int array
    """
    with torch.no_grad():
        feats, fg_f, bg_f, _, _, _ = model(pc_tensor)
        ft   = feats.permute(0,2,1)                          # (1,N,D)
        fg_p = fg_f.mean(2, keepdim=True).permute(0,2,1)    # (1,1,D)
        bg_p = bg_f.mean(2, keepdim=True).permute(0,2,1)    # (1,1,D)
        pred = ((ft-fg_p).norm(-1) < (ft-bg_p).norm(-1)).long().squeeze(0)  # (N,)
    return pred.cpu().numpy()


def run_robustness_test(ckpt_path, drop_rates=[0.0, 0.1, 0.2], n_samples=100, seed=99):
    """
    For each drop_rate:
      1. Run inference on FULL point cloud → pred_full
      2. Drop drop_rate fraction of points randomly
      3. Run inference on REDUCED point cloud → pred_drop
      4. Measure consistency = IOU(pred_full[kept], pred_drop)
      5. Measure IOU vs ground truth for both
    """
    model = CoSegNet(CFG['n_fg'], CFG['n_bg'], CFG['dgcnn_k'], CFG['emb_dim'], CFG['n_parts']).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    test_ds = ShapeNet_coseg('test', CFG['num_points'], CFG['obj_class'])
    n_samples = min(n_samples, len(test_ds))

    np.random.seed(seed)
    results = {}

    for drop_rate in drop_rates:
        consistency_list = []
        iou_full_list    = []
        iou_drop_list    = []

        for i in range(n_samples):
            pc, lbl = test_ds[i]                          # (N,3), (N,)
            N = pc.shape[0]
            pc_t = torch.tensor(pc).unsqueeze(0).to(DEVICE)   # (1,N,3)

            # Full prediction
            pred_full = predict_on_cloud(model, pc_t)      # (N,)
            iou_full  = jaccard_score(lbl, pred_full, average='binary', zero_division=0)
            iou_full_list.append(iou_full)

            if drop_rate == 0.0:
                consistency_list.append(1.0)
                iou_drop_list.append(iou_full)
                continue

            # Drop points
            n_keep   = int(N * (1.0 - drop_rate))
            keep_idx = np.sort(np.random.choice(N, n_keep, replace=False))
            pc_drop  = torch.tensor(pc[keep_idx]).unsqueeze(0).to(DEVICE)  # (1,n_keep,3)

            pred_drop = predict_on_cloud(model, pc_drop)   # (n_keep,)

            # Consistency: compare predictions on the SAME kept points
            consistency = jaccard_score(
                pred_full[keep_idx], pred_drop,
                average='binary', zero_division=0)
            consistency_list.append(consistency)

            # IOU vs ground truth on the kept points only
            iou_drop = jaccard_score(
                lbl[keep_idx], pred_drop,
                average='binary', zero_division=0)
            iou_drop_list.append(iou_drop)

        results[drop_rate] = {
            'consistency':     float(np.mean(consistency_list)),
            'consistency_std': float(np.std(consistency_list)),
            'iou_full':        float(np.mean(iou_full_list)),
            'iou_drop':        float(np.mean(iou_drop_list)),
            'iou_drop_std':    float(np.std(iou_drop_list)),
        }

        print(f'Drop {int(drop_rate*100):2d}% '
              f'| Consistency={results[drop_rate]["consistency"]:.4f} (±{results[drop_rate]["consistency_std"]:.3f}) '
              f'| IOU(full)={results[drop_rate]["iou_full"]:.4f} '
              f'| IOU(drop)={results[drop_rate]["iou_drop"]:.4f}')

    return results


print('Robustness test function defined.')
print('Running on 100 test samples. This takes ~5 minutes...')
rob_results = run_robustness_test(CKPT_PATH, drop_rates=[0.0, 0.1, 0.2], n_samples=100)

In [ ]:
# ── CELL 11: Robustness results table + plot ──────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

drop_rates   = sorted(rob_results.keys())
consistency  = [rob_results[d]['consistency']  for d in drop_rates]
iou_full     = [rob_results[d]['iou_full']     for d in drop_rates]
iou_drop     = [rob_results[d]['iou_drop']     for d in drop_rates]
cons_std     = [rob_results[d]['consistency_std'] for d in drop_rates]

drop_pct = [int(d*100) for d in drop_rates]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Task 5 — Robustness to Point Dropout  (n=100 test shapes)',
             fontsize=13, fontweight='bold')

# Plot 1: Consistency
axes[0].bar(drop_pct, consistency, width=6, color=['steelblue','darkorange','tomato'],
            edgecolor='black', linewidth=0.7)
for i, (x, y, s) in enumerate(zip(drop_pct, consistency, cons_std)):
    axes[0].errorbar(x, y, yerr=s, fmt='none', color='black', capsize=5, lw=1.5)
    axes[0].text(x, y + 0.01, f'{y:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Points Dropped (%)', fontsize=11)
axes[0].set_ylabel('Consistency (IOU with full prediction)', fontsize=11)
axes[0].set_title('Prediction Consistency after Dropout')
axes[0].set_ylim(0, 1.1)
axes[0].set_xticks(drop_pct)
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: IOU vs GT
x = np.arange(len(drop_rates))
w = 0.35
axes[1].bar(x - w/2, iou_full, width=w, label='IOU vs GT (full pts)',  color='steelblue', edgecolor='black', lw=0.7)
axes[1].bar(x + w/2, iou_drop, width=w, label='IOU vs GT (kept pts)',  color='darkorange', edgecolor='black', lw=0.7)
for i, (a, b) in enumerate(zip(iou_full, iou_drop)):
    axes[1].text(i - w/2, a + 0.005, f'{a:.3f}', ha='center', va='bottom', fontsize=9)
    axes[1].text(i + w/2, b + 0.005, f'{b:.3f}', ha='center', va='bottom', fontsize=9)
axes[1].set_xlabel('Points Dropped (%)', fontsize=11)
axes[1].set_ylabel('IOU vs Ground Truth', fontsize=11)
axes[1].set_title('Segmentation Quality after Dropout')
axes[1].set_xticks(x); axes[1].set_xticklabels([f'{p}%' for p in drop_pct])
axes[1].set_ylim(0, max(iou_full)*1.2)
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results/task5_robustness_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*65)
print(f'  {"Drop %":<8} {"Consistency":>13} {"IOU(full)": >12} {"IOU(drop)":>12}')
print('='*65)
for d in drop_rates:
    r = rob_results[d]
    print(f'  {int(d*100):<7}% {r["consistency"]:>13.4f} {r["iou_full"]:>12.4f} {r["iou_drop"]:>12.4f}')
print('='*65)

# Interpretation
c0 = rob_results[0.0]['consistency']
c1 = rob_results[0.1]['consistency']
c2 = rob_results[0.2]['consistency']
drop_pct_10 = (1 - c1/c0) * 100
drop_pct_20 = (1 - c2/c0) * 100
print(f'\nConsistency drop: 10% removal → {drop_pct_10:.1f}% degradation')
print(f'Consistency drop: 20% removal → {drop_pct_20:.1f}% degradation')

In [ ]:
# ── CELL 12: Visualisation — 4 shapes, 3 views each ──────────────────────────
# Shows: Ground Truth | Full prediction | 10% dropped | 20% dropped
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

model_loaded = CoSegNet(CFG['n_fg'], CFG['n_bg'], CFG['dgcnn_k'], CFG['emb_dim'], CFG['n_parts']).to(DEVICE)
model_loaded.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model_loaded.eval()

test_ds_viz = ShapeNet_coseg('test', CFG['num_points'], CFG['obj_class'])

# Pick 4 diverse samples
sample_indices = [0, 5, 12, 25]
N_COLS = 4   # GT | full | drop10 | drop20
N_ROWS = len(sample_indices)
col_titles = ['Ground Truth', 'Full prediction', '10% dropped', '20% dropped']

fig = plt.figure(figsize=(20, 5 * N_ROWS))
fig.suptitle('Task 5 — Segmentation Robustness Visualisation', fontsize=14, fontweight='bold')

COLORS = np.array([[70, 130, 180], [220, 80, 80]], dtype='float32') / 255.0  # blue=fg, red=bg

np.random.seed(42)

for row, s_idx in enumerate(sample_indices):
    pc, lbl = test_ds_viz[s_idx]
    N = pc.shape[0]
    pc_t = torch.tensor(pc).unsqueeze(0).to(DEVICE)

    pred_full = predict_on_cloud(model_loaded, pc_t)

    iou_full = jaccard_score(lbl, pred_full, average='binary', zero_division=0)

    views_pc   = [pc, pc]
    views_lbl  = [lbl, pred_full]
    view_ious  = ['GT', f'IOU={iou_full:.3f}']

    for drop_rate in [0.1, 0.2]:
        n_keep   = int(N * (1 - drop_rate))
        keep_idx = np.sort(np.random.choice(N, n_keep, replace=False))
        pc_drop  = torch.tensor(pc[keep_idx]).unsqueeze(0).to(DEVICE)
        pred_drop = predict_on_cloud(model_loaded, pc_drop)
        iou_drop = jaccard_score(lbl[keep_idx], pred_drop, average='binary', zero_division=0)
        consist  = jaccard_score(pred_full[keep_idx], pred_drop, average='binary', zero_division=0)
        views_pc.append(pc[keep_idx])
        views_lbl.append(pred_drop)
        view_ious.append(f'IOU={iou_drop:.3f} | Cons={consist:.3f}')

    for col in range(N_COLS):
        ax = fig.add_subplot(N_ROWS, N_COLS, row*N_COLS + col + 1, projection='3d')
        pts = views_pc[col]
        c   = COLORS[views_lbl[col]]
        ax.scatter(pts[:,0], pts[:,1], pts[:,2], c=c, s=1.5, depthshade=False)
        if row == 0:
            ax.set_title(col_titles[col], fontsize=11, fontweight='bold')
        ax.set_xlabel(view_ious[col], fontsize=8)
        ax.set_axis_off()

plt.tight_layout()
plt.savefig('/content/results/task5_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: task5_visualization.png')

In [ ]:
# ── CELL 13: Complete final results summary (all tasks) ───────────────────────
import matplotlib.pyplot as plt

print('\n' + '='*70)
print('  COMPLETE BTP RESULTS — ALL TASKS')
print('  Dataset: ShapeNet Part — Chair | 1024 pts | DGCNN + PointSamplers')
print('='*70)

sections = [
    ('Core Results (v5)', [
        ('Baseline',                       0.4031, 0.6050),
        ('+ Spatial uniform λ=0.01 k=10',  0.4635, 0.6670),
        ('+ Spatial weighted σ=1',         0.3877, 0.5880),
        ('+ Entropy λ=0.0001  ← BEST',     0.4885, 0.6906),
        ('Full model (all losses)',         0.3900, 0.5905),
    ]),
    ('Task 1 — σ Ablation (weighted spatial)', [
        ('σ=0.05',  0.3581, 0.5273),
        ('σ=0.1',   0.3611, 0.5306),
        ('σ=0.5',   0.3818, 0.5526),
        ('σ=1.0  ← best sigma', 0.3887, 0.5598),
        ('σ=2.0',   0.3464, 0.5145),
    ]),
    ('Task 2 — Entropy at 50 epochs', [
        ('Entropy 15 ep (best)',  0.4885, 0.6906),
        ('Entropy 50 ep',         0.3864, 0.5574),
    ]),
    ('Task 3 — Balance Loss', [
        ('Entropy only (no balance)',  0.4885, 0.6906),
        ('+ Balance λ=0.001',          0.3639, 0.5336),
        ('+ Balance λ=0.01  ← best',   0.3831, 0.5540),
        ('+ Balance λ=0.05',           0.3522, 0.5210),
        ('+ Balance λ=0.1',            0.3533, 0.5221),
    ]),
    ('Task 4 — L2 Feature Normalization', [
        ('Without L2 norm',  0.3875, 0.5586),
        ('With L2 norm',     0.3504, 0.5190),
    ]),
]

for section_name, rows in sections:
    print(f'\n  {section_name}')
    print(f'  {"-"*65}')
    print(f'  {"Method":<40} {"IOU":>8} {"F1":>8} {"ΔIOU":>8}')
    for name, iou, f1 in rows:
        delta = iou - 0.4031
        print(f'  {name:<40} {iou:>8.4f} {f1:>8.4f} {delta:>+8.4f}')

print(f'\n  Task 5 — Robustness')
print(f'  {"-"*65}')
for d in sorted(rob_results.keys()):
    r = rob_results[d]
    print(f'  {int(d*100):2d}% dropped  |  Consistency={r["consistency"]:.4f}  '
          f'IOU(full)={r["iou_full"]:.4f}  IOU(drop)={r["iou_drop"]:.4f}')

print('\n' + '='*70)

In [ ]:
# ── CELL 14: Save everything to Google Drive ──────────────────────────────────
import shutil, os

dest = '/content/drive/MyDrive/BTP_coseg_v6_final'
os.makedirs(dest, exist_ok=True)
shutil.copytree('/content/results', dest, dirs_exist_ok=True)

print('Saved to Drive:')
for root, dirs, files in os.walk(dest):
    for f in files:
        print(' ', os.path.join(root,f).replace(dest,''))